# BUSM131 — Telco Customer Churn: Cox Proportional Hazards Model

Semi-parametric Cox PH survival regression.

**Contents:** Cox PH fitting, hazard ratios, PH assumption testing, partial effects, customer risk segmentation (4 tiers), survival predictions, business recommendations.


In [ ]:
# Run this first if lifelines is not installed:
# !pip install lifelines

%matplotlib inline


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings, os
warnings.filterwarnings('ignore')

from lifelines import CoxPHFitter, KaplanMeierFitter
from sklearn.preprocessing import StandardScaler

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 7)
plt.rcParams['font.size'] = 12


## 1. DATA PREPARATION FOR COX MODEL


In [ ]:
print("=" * 65)


In [ ]:
print("=" * 65)

# *** UPDATE THIS PATH to your local file location ***
df = pd.read_csv('WA_Fn-UseC_-Telco-Customer-Churn.csv')

df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df['TotalCharges'] = df['TotalCharges'].fillna(0)

# Survival variables
df['T'] = df['tenure']
df['E'] = (df['Churn'] == 'Yes').astype(int)
df.loc[df['T'] == 0, 'T'] = 0.5

# Create dummy variables for categorical features
cat_cols = ['gender', 'Partner', 'Dependents', 'PhoneService',
            'MultipleLines', 'InternetService', 'OnlineSecurity',
            'OnlineBackup', 'DeviceProtection', 'TechSupport',
            'StreamingTV', 'StreamingMovies', 'Contract',
            'PaperlessBilling', 'PaymentMethod']

cox_df = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# Drop columns not needed for Cox model
# TotalCharges is highly collinear with tenure — drop it
drop_cols = ['customerID', 'Churn', 'TotalCharges', 'tenure']
cox_data = cox_df.drop(columns=[c for c in drop_cols if c in cox_df.columns])

# Standardise MonthlyCharges for better convergence
scaler = StandardScaler()
cox_data['MonthlyCharges'] = scaler.fit_transform(cox_data[['MonthlyCharges']])

print(f"Cox model features: {cox_data.shape[1] - 2} covariates + T + E")
print(f"Sample size: {cox_data.shape[0]}")


## 2. FIT COX PH MODEL


In [ ]:
print("\n" + "-" * 65)


### 2. COX PH MODEL FITTING


In [ ]:
print("-" * 65)

cph = CoxPHFitter(penalizer=0.01)  # small L2 penalty for stability
cph.fit(cox_data, duration_col='T', event_col='E')

print(f"\n  Concordance Index: {cph.concordance_index_:.4f}")
print(f"  Partial AIC:       {cph.AIC_partial_:.2f}")
ll_test = cph.log_likelihood_ratio_test()
print(f"  Log-Likelihood Ratio Test: χ²={ll_test.test_statistic:.2f}, "
      f"p={ll_test.p_value:.2e}")


## 3. MODEL SUMMARY — HAZARD RATIOS


In [ ]:
print("\n" + "-" * 65)


### 3. HAZARD RATIOS — FULL SUMMARY


In [ ]:
print("-" * 65)

cph.print_summary(columns=['coef', 'exp(coef)', 'se(coef)', 'p',
                            'exp(coef) lower 95%', 'exp(coef) upper 95%'])

# Top factors by magnitude
summary = cph.summary.copy()
summary['abs_coef'] = summary['coef'].abs()
summary = summary.sort_values('abs_coef', ascending=False)

print("\n  TOP 15 HAZARD FACTORS (sorted by |coefficient|):")
print(summary[['coef', 'exp(coef)', 'p']].head(15).to_string())

# Interpretation table
print("\n" + "-" * 65)
print("  HAZARD RATIO INTERPRETATION:")
print("-" * 65)
print("""
  exp(coef) > 1  →  INCREASES churn hazard (bad for retention)
  exp(coef) < 1  →  DECREASES churn hazard (good for retention)
  exp(coef) = 1  →  No effect
  
  Example: exp(coef)=1.50 means 50% higher hazard of churning.
           exp(coef)=0.60 means 40% lower hazard of churning.
""")

for idx, row in summary.head(10).iterrows():
    direction = "INCREASES" if row['coef'] > 0 else "DECREASES"
    pct = abs(row['exp(coef)'] - 1) * 100
    sig = '***' if row['p'] < 0.001 else ('**' if row['p'] < 0.01 else
          ('*' if row['p'] < 0.05 else 'n.s.'))
    print(f"  {idx:45s}  {direction} hazard by {pct:5.1f}%  {sig}")


## 4. PLOTS


In [ ]:
print("\n" + "-" * 65)


### 4. GENERATING PLOTS


In [ ]:
print("-" * 65)


**Fig 21: Forest plot of hazard ratios**


In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
cph.plot(ax=ax, hazard_ratios=True)
ax.axvline(x=1, color='red', linestyle='--', alpha=0.6)
ax.set_title('Cox PH Model — Hazard Ratios with 95% CI',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


**Fig 22: Top 15 hazard ratios bar chart**


In [ ]:
top15 = summary.head(15)[['exp(coef)']].copy()
top15['color'] = ['#e74c3c' if v > 1 else '#2ecc71'
                  for v in top15['exp(coef)']]

fig, ax = plt.subplots(figsize=(12, 8))
bars = ax.barh(range(len(top15)), top15['exp(coef)'],
               color=top15['color'], edgecolor='black', height=0.7)
ax.set_yticks(range(len(top15)))
ax.set_yticklabels(top15.index, fontsize=10)
ax.axvline(x=1, color='black', linewidth=1.5, linestyle='-')
ax.set_xlabel('Hazard Ratio (exp(coef))', fontsize=13)
ax.set_title('Top 15 Factors — Cox PH Hazard Ratios',
             fontsize=14, fontweight='bold')
for bar, val in zip(bars, top15['exp(coef)']):
    ax.text(val + 0.02, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontsize=9)
ax.invert_yaxis()
plt.tight_layout()
plt.show()


## 5. PROPORTIONAL HAZARDS ASSUMPTION TEST


In [ ]:
print("\n" + "-" * 65)


In [ ]:
print("-" * 65)

try:
    ph_results = cph.check_assumptions(cox_data, p_value_threshold=0.05,
                                        show_plots=False)
    print("  PH assumption test completed — see output above.")
    print("  Variables that violate PH assumption may need stratification")
    print("  or time-varying coefficients.")
except Exception as e:
    print(f"  PH test note: {e}")
    print("  Some variables may violate the PH assumption.")
    print("  Consider: stratified Cox model or time-varying covariates.")


## 6. PARTIAL EFFECTS ON SURVIVAL


In [ ]:
print("\n" + "-" * 65)


In [ ]:
print("-" * 65)

# Contract type effect
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Contract: Two year
try:
    cph.plot_partial_effects_on_outcome('Contract_Two year', [0, 1],
                                         ax=axes[0, 0])
    axes[0, 0].set_title('Effect of Two-Year Contract', fontweight='bold')
except:
    axes[0, 0].text(0.5, 0.5, 'Plot unavailable', ha='center', transform=axes[0,0].transAxes)

# Internet: Fiber optic
try:
    cph.plot_partial_effects_on_outcome('InternetService_Fiber optic', [0, 1],
                                         ax=axes[0, 1])
    axes[0, 1].set_title('Effect of Fiber Optic Internet', fontweight='bold')
except:
    axes[0, 1].text(0.5, 0.5, 'Plot unavailable', ha='center', transform=axes[0,1].transAxes)

# Payment: Electronic check
try:
    cph.plot_partial_effects_on_outcome('PaymentMethod_Electronic check', [0, 1],
                                         ax=axes[1, 0])
    axes[1, 0].set_title('Effect of Electronic Check Payment', fontweight='bold')
except:
    axes[1, 0].text(0.5, 0.5, 'Plot unavailable', ha='center', transform=axes[1,0].transAxes)

# Monthly charges (continuous)
try:
    cph.plot_partial_effects_on_outcome('MonthlyCharges',
        [cox_data['MonthlyCharges'].quantile(0.25),
         cox_data['MonthlyCharges'].median(),
         cox_data['MonthlyCharges'].quantile(0.75)],
        ax=axes[1, 1])
    axes[1, 1].set_title('Effect of Monthly Charges (quartiles)', fontweight='bold')
except:
    axes[1, 1].text(0.5, 0.5, 'Plot unavailable', ha='center', transform=axes[1,1].transAxes)

plt.suptitle('Partial Effects on Survival Probability',
             fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()


## 7. CUSTOMER RISK SCORING & SEGMENTATION


In [ ]:
print("\n" + "-" * 65)


In [ ]:
print("-" * 65)

# Predict partial hazard for each customer
cox_features = cox_data.drop(columns=['T', 'E'])
df['hazard_score'] = cph.predict_partial_hazard(cox_features).values

# Predict median survival time
df['predicted_median_survival'] = cph.predict_median(cox_features).values

# Segment into risk groups
df['risk_segment'] = pd.qcut(df['hazard_score'], q=4,
                              labels=['Low Risk', 'Medium Risk',
                                      'High Risk', 'Very High Risk'])

print("\nRisk Segment Summary:")
risk_summary = df.groupby('risk_segment', observed=True).agg(
    n=('E', 'size'),
    churned=('E', 'sum'),
    churn_rate=('E', 'mean'),
    avg_tenure=('T', 'mean'),
    avg_monthly=('MonthlyCharges', 'mean'),
    median_predicted_survival=('predicted_median_survival', 'median')
).round(2)
risk_summary['churn_rate'] = (risk_summary['churn_rate'] * 100).round(1)
print(risk_summary.to_string())
risk_summary.to_csv('cox_risk_segments.csv')


**Fig 24: KM curves by risk segment**


In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
risk_colors = {'Low Risk': '#2ecc71', 'Medium Risk': '#f39c12',
               'High Risk': '#e67e22', 'Very High Risk': '#e74c3c'}

for segment, color in risk_colors.items():
    mask = df['risk_segment'] == segment
    kmf_g = KaplanMeierFitter()
    kmf_g.fit(df.loc[mask, 'T'], df.loc[mask, 'E'], label=segment)
    kmf_g.plot_survival_function(ax=ax, ci_show=False, color=color, linewidth=2.5)

ax.set_title('Survival Curves by Cox PH Risk Segment',
             fontsize=16, fontweight='bold')
ax.set_xlabel('Tenure (Months)', fontsize=13)
ax.set_ylabel('Survival Probability', fontsize=13)
ax.legend(fontsize=12, title='Risk Segment')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


**Fig 25: Hazard score distribution**


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for label, event, color in [('Churned', 1, '#e74c3c'), ('Active', 0, '#2ecc71')]:
    mask = df['E'] == event
    ax.hist(df.loc[mask, 'hazard_score'], bins=50, alpha=0.6,
            color=color, label=label, edgecolor='black', linewidth=0.5)
ax.set_title('Distribution of Cox Hazard Scores by Actual Outcome',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Partial Hazard Score')
ax.set_ylabel('Count')
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()


## 8. SURVIVAL PREDICTIONS FOR CUSTOMER PROFILES


In [ ]:
print("\n" + "-" * 65)


In [ ]:
print("-" * 65)


In [ ]:
# Create median baseline profile
base = cox_features.median()


In [ ]:
profiles = {
    'High-Risk:\nMonth-to-month, Fiber optic,\nElectronic check, No security': {
        'Contract_One year': 0, 'Contract_Two year': 0,
        'InternetService_Fiber optic': 1, 'InternetService_No': 0,
        'PaymentMethod_Electronic check': 1,
        'OnlineSecurity_Yes': 0, 'TechSupport_Yes': 0,
    },
    'Medium-Risk:\nOne-year contract, DSL,\nAuto-pay, Has security': {
        'Contract_One year': 1, 'Contract_Two year': 0,
        'InternetService_Fiber optic': 0, 'InternetService_No': 0,
        'PaymentMethod_Electronic check': 0,
        'OnlineSecurity_Yes': 1, 'TechSupport_Yes': 1,
    },
    'Low-Risk:\nTwo-year contract, DSL,\nAuto-pay, Has security': {
        'Contract_One year': 0, 'Contract_Two year': 1,
        'InternetService_Fiber optic': 0, 'InternetService_No': 0,
        'PaymentMethod_Electronic check': 0,
        'OnlineSecurity_Yes': 1, 'TechSupport_Yes': 1,
    }
}


In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
profile_colors = ['#e74c3c', '#f39c12', '#2ecc71']


In [ ]:
for (name, overrides), color in zip(profiles.items(), profile_colors):
    profile = base.copy()
    for k, v in overrides.items():
        if k in profile.index:
            profile[k] = v
    profile_df = pd.DataFrame([profile])
    
    sf = cph.predict_survival_function(profile_df)
    median_t = float(np.ravel(cph.predict_median(profile_df))[0])
    
    ax.plot(sf.index, sf.values.flatten(), color=color, lw=2.5, label=name)
    print(f"  {name.replace(chr(10),' ')}")
    print(f"    Predicted median survival: {median_t:.1f} months")
    print(f"    P(survive > 12 months): {sf.loc[12].values[0]:.3f}")
    print(f"    P(survive > 24 months): {sf.loc[24].values[0]:.3f}")
    print()


In [ ]:
ax.axhline(y=0.5, color='grey', linestyle='--', alpha=0.5, label='50% survival')
ax.set_title('Predicted Survival Curves for Customer Profiles\n(Cox PH Model)',
             fontsize=14, fontweight='bold')
ax.set_xlabel('Months', fontsize=13)
ax.set_ylabel('Survival Probability', fontsize=13)
ax.legend(fontsize=9, loc='lower left')
ax.grid(True, alpha=0.3)
ax.set_xlim(0, 75)
plt.tight_layout()
plt.show()


## 9. BUSINESS RECOMMENDATIONS


In [ ]:
print("\n" + "=" * 65)


In [ ]:
print("=" * 65)
print(f"""
╔═════════════════════════════════════════════════════════════════╗
║  COX PH MODEL RESULTS SUMMARY                                  ║
║                                                                  ║
║  Concordance Index: {cph.concordance_index_:.4f}                               ║
║  (1.0 = perfect, 0.5 = random)                                  ║
╚═════════════════════════════════════════════════════════════════╝
KEY FINDINGS FOR THE REPORT:
────────────────────────────
1. SURVIVAL ANALYSIS adds a TIME dimension that standard classifiers
   miss — it answers "WHEN will a customer churn?" not just "WILL
   they churn?" This allows the business to prioritise interventions
   by urgency.
2. The Cox PH model achieves concordance of {cph.concordance_index_:.3f}, indicating
   strong ability to rank customers by churn timing.
3. TOP CHURN ACCELERATORS (hazard ratio > 1):
   - Month-to-month contracts (vs baseline)
   - Fiber optic internet service
   - Electronic check payment
   - No online security / tech support
4. TOP CHURN PROTECTORS (hazard ratio < 1):
   - Two-year contracts (strongest protector)
   - One-year contracts
   - Automatic payment methods (bank transfer, credit card)
5. ACTIONABLE RECOMMENDATIONS:
   a) Target "Very High Risk" segment (~25% of customers, highest
      churn rate) with personalised retention offers
   b) Incentivise month-to-month customers to upgrade to annual
      or two-year contracts (strongest single factor)
   c) Investigate fiber optic service quality — high monthly charges
      and high churn suggest potential value-for-money issues
   d) Encourage electronic check users to switch to automatic payment
   e) Bundle online security and tech support into standard packages
   f) Focus retention efforts in the first 12 months — this is when
      the highest-risk customers are most likely to leave
6. COMPARISON WITH RANDOM FOREST:
   - Random Forest classifies churn (Yes/No) with ROC-AUC ~0.84
   - Cox PH models time-to-churn with concordance ~{cph.concordance_index_:.2f}
   - Both agree on key risk factors (contract, internet, payment)
   - Combined approach: use RF for immediate targeting, Cox for
     timing and urgency prioritisation
""")


In [ ]:
print("=" * 65)


## COX PH SURVIVAL ANALYSIS COMPLETE


In [ ]:
print("=" * 65)
